# Spike #17 — Un modèle zero-shot classe-t-il une tendance par famille de goût ?

**Question** : un modèle de classification zero-shot classe-t-il correctement une tendance par famille de goût ou d'odeur ?
**Méthode du ticket** : 20 tendances réelles étiquetées à la main **par le Product**, comparées aux classes du modèle. **Seuil** : 80 % sur 20 exemples.

⚠️ **État au 12/11/2026** : l'étiquetage Product n'est pas encore fait. Les étiquettes utilisées ici sont une **proposition Data**, à remplacer par celles du Product (colonne `etiquette_product` de `data/spike17/tendances_familles.csv`). Le résultat ci-dessous est donc **provisoire** et ne valide pas le seuil.

Deux modèles :
- **A. LLM local en zero-shot** (qwen3.5:4b via Ollama, choisi dans le benchmark), contraint à répondre par une des 10 familles. Exécuté le 12/11/2026, résultats dans `data/spike17/spike17_zero_shot_ollama.json`.
- **B. Modèle Hugging Face d'inférence NLI** (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`, multilingue), comme demandé dans le ticket. À exécuter sur un poste avec `pip install transformers torch` (téléchargement du modèle ≈ 1 Go). La cellule s'arrête proprement si la bibliothèque est absente.

In [1]:
import json, pandas as pd
pd.set_option("display.width", 160)
ref = pd.read_csv("data/spike17/tendances_familles.csv", encoding="utf-8-sig")
FAMILLES = json.load(open("data/spike17/spike17_zero_shot_ollama.json", encoding="utf-8"))["familles"]
# étiquette de référence : celle du Product si remplie, sinon la proposition Data
ref["reference"] = ref["etiquette_product"].fillna("").where(ref["etiquette_product"].fillna("") != "", ref["etiquette_proposee_data"])
ref["source_reference"] = ["Product" if str(p).strip() else "proposition Data" for p in ref["etiquette_product"].fillna("")]
ref

,tendance,etiquette_proposee_data,etiquette_product,ambigu,commentaire,reference,source_reference
0,matcha,végétal / herbacé,NaN,non,thé vert en poudre : notes herbacées et umami,végétal / herbacé,proposition Data
1,yuzu,agrumes,NaN,non,NaN,agrumes,proposition Data
2,ube,sucré / confiserie,NaN,oui,igname violette ; usages surtout desserts (not...,sucré / confiserie,proposition Data
3,oat milk,lacté / crémeux,NaN,non,NaN,lacté / crémeux,proposition Data
4,kombucha,fermenté / umami,NaN,non,NaN,fermenté / umami,proposition Data
5,cold brew coffee,torréfié / café-cacao,NaN,non,NaN,torréfié / café-cacao,proposition Data
6,birria,épicé / piquant,NaN,oui,ragoût mexicain aux piments ; aussi umami / vi...,épicé / piquant,proposition Data
7,gochujang,épicé / piquant,NaN,oui,pâte de piment fermentée : épicé + fermenté,épicé / piquant,proposition Data
8,hot honey,épicé / piquant,NaN,oui,miel pimenté : épicé + sucré,épicé / piquant,proposition Data
9,mochi,sucré / confiserie,NaN,non,NaN,sucré / confiserie,proposition Data


## A. LLM local zero-shot (qwen3.5:4b, Ollama)

In [2]:
o = pd.DataFrame(json.load(open("data/spike17/spike17_zero_shot_ollama.json", encoding="utf-8"))["resultats"])
a = ref.merge(o[["tendance", "prediction", "secondes"]], on="tendance")
a["correct"] = a.prediction == a.reference
print(f"Bonnes familles : {a.correct.sum()} / {len(a)} ({a.correct.mean():.0%}) — seuil 80 % — référence : {a.source_reference.iloc[0]}")
a[["tendance", "reference", "prediction", "correct", "ambigu"]]

Bonnes familles : 15 / 20 (75%) — seuil 80 % — référence : proposition Data


,tendance,reference,prediction,correct,ambigu
0,matcha,végétal / herbacé,torréfié / café-cacao,False,non
1,yuzu,agrumes,agrumes,True,non
2,ube,sucré / confiserie,fruité,False,oui
3,oat milk,lacté / crémeux,lacté / crémeux,True,non
4,kombucha,fermenté / umami,fermenté / umami,True,non
5,cold brew coffee,torréfié / café-cacao,torréfié / café-cacao,True,non
6,birria,épicé / piquant,épicé / piquant,True,oui
7,gochujang,épicé / piquant,épicé / piquant,True,oui
8,hot honey,épicé / piquant,épicé / piquant,True,oui
9,mochi,sucré / confiserie,sucré / confiserie,True,non


In [3]:
# Erreurs : vraies erreurs ou tendances qui relèvent de plusieurs familles ?
a[~a.correct][["tendance", "reference", "prediction", "ambigu", "commentaire"]]

,tendance,reference,prediction,ambigu,commentaire
0,matcha,végétal / herbacé,torréfié / café-cacao,non,thé vert en poudre : notes herbacées et umami
2,ube,sucré / confiserie,fruité,oui,igname violette ; usages surtout desserts (not...
11,Dubai chocolate,fruits à coque,torréfié / café-cacao,oui,chocolat + crème de pistache + kadayif
14,lavender latte,floral,lacté / crémeux,oui,lavande + lait
16,black sesame,fruits à coque,torréfié / café-cacao,oui,graine torréfiée : fruits à coque + torréfié


## B. Modèle Hugging Face zero-shot (NLI)

In [4]:
try:
    from transformers import pipeline
except ImportError:
    print("transformers non installé : lancer `pip install transformers torch sentencepiece` puis relancer cette cellule.")
else:
    clf = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
    preds = []
    for t in ref.tendance:
        r = clf(t, FAMILLES, hypothesis_template="Le goût de cette tendance alimentaire est {}.", multi_label=False)
        preds.append({"tendance": t, "prediction_hf": r["labels"][0], "score_hf": round(r["scores"][0], 3)})
    b = ref.merge(pd.DataFrame(preds), on="tendance")
    b["correct"] = b.prediction_hf == b.reference
    print(f"Bonnes familles (HF) : {b.correct.sum()} / {len(b)} ({b.correct.mean():.0%})")
    display(b[["tendance", "reference", "prediction_hf", "score_hf", "correct"]])

transformers non installé : lancer `pip install transformers torch sentencepiece` puis relancer cette cellule.


## Décision provisoire

**NO-GO en l'état pour une classification à une seule famille, à confirmer avec l'étiquetage Product.**

- Le LLM local obtient **15 / 20 (75 %)** contre la proposition Data, sous le seuil de 80 %.
- Sur les 5 erreurs, **une seule est franche** : matcha classé « torréfié » au lieu de « végétal / herbacé ». Les 4 autres sont des tendances qui relèvent réellement de plusieurs familles : Dubai chocolate (cacao + pistache), sésame noir (fruits à coque + torréfié), latte lavande (floral + lacté), ube (sucré + fruité ?).
- **Le problème vient autant de la question que du modèle** : une seule famille par tendance est trop réducteur pour des arômes. Il faut passer en **multi-étiquettes** (1 à 3 familles par tendance) et mesurer la précision sur les étiquettes du Product.

**Suite** : (1) le Product remplit `etiquette_product` (une ou plusieurs familles) ; (2) relancer ce notebook ; (3) exécuter la partie B sur un poste avec `transformers` pour comparer au modèle Hugging Face demandé par le ticket.